# Organizing the public Python library


In [1]:
from pathlib import Path
import ast
import re
import sys
import tomllib

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display


def display(value):
    module = value.__class__.__module__
    if module.startswith("pandas") and hasattr(value, "_repr_html_"):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + "</div>"))
    return _display(value)


BOOK_ROOT = Path.cwd().parents[1]
import os
import RiskLabAI
PYTHON_PACKAGE = Path(RiskLabAI.__file__).resolve().parent
PYTHON_SOURCE = PYTHON_PACKAGE.parent
JULIA_ROOT_VALUE = os.environ.get("RISKLABAI_JULIA_ROOT", "")
JULIA_PACKAGE = Path(JULIA_ROOT_VALUE).expanduser() if JULIA_ROOT_VALUE else None
TABLES = BOOK_ROOT / "outputs" / "tables" / "A_risklabai_design"
FIGURES = BOOK_ROOT / "outputs" / "figures" / "A_risklabai_design"
BOOK_FIGURES = FIGURES
FIGURE_DATA = BOOK_ROOT / "outputs" / "figure_data" / "A_risklabai_design"
for directory in [TABLES, FIGURES, BOOK_FIGURES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(BOOK_ROOT))
from book_figure_style import PALETTE, apply_book_figure_style, finish_axis

apply_book_figure_style()


def rendered_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (bool, np.bool_)):
        return "Pass" if bool(value) else "Fail"
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if isinstance(value, (float, np.floating)) and float(value).is_integer():
        return f"{int(value):,}"
    return str(value)


def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))


def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f"{stem}.csv", index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position="htbp")
    if fit:
        latex = latex.replace("\\begin{tabular}", "\\centering\n\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}\\end{adjustbox}", 1)
    else:
        latex = latex.replace("\\begin{tabular}", "\\centering\n\\begin{tabular}", 1)
    (TABLES / f"{stem}.tex").write_text(latex, encoding="utf-8")
    display(shown)
    return shown


def save_figure(figure, stem):
    for directory in [FIGURES, BOOK_FIGURES]:
        figure.savefig(directory / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor=PALETTE["white"], transparent=False)
        figure.savefig(directory / f"{stem}.pdf", bbox_inches="tight", facecolor=PALETTE["white"], transparent=False)
    display(figure)
    plt.close(figure)


def add_box(axis, x, y, width, height, text, facecolor, edgecolor, textcolor=None):
    patch = FancyBboxPatch((x - width / 2, y - height / 2), width, height, boxstyle="round,pad=0.04,rounding_size=0.08", facecolor=facecolor, edgecolor=edgecolor, linewidth=1.3, zorder=3)
    axis.add_patch(patch)
    axis.text(x, y, text, ha="center", va="center", color=textcolor or PALETTE["navy"], fontsize=10.2, zorder=4)


import RiskLabAI
assert RiskLabAI.__version__ == "3.2.0"


## Public entry points


In [2]:
root_source = (PYTHON_PACKAGE / "__init__.py").read_text(encoding="utf-8")
source_modules = sorted(path for path in PYTHON_PACKAGE.rglob("*.py") if "__pycache__" not in path.parts)
public_names = list(RiskLabAI.__all__)
public_areas = [name for name in public_names if name != "__version__"]
structure = pd.DataFrame([
    {"Design choice": "One public front door", "Public evidence": "RiskLabAI.__all__", "Purpose": "Shows supported starting points"},
    {"Design choice": "Modules loaded when requested", "Public evidence": "Module __getattr__", "Purpose": "Delays optional and heavier imports"},
    {"Design choice": "Grouped subject areas", "Public evidence": f"{len(public_areas)} exported areas", "Purpose": "Keeps related methods together"},
    {"Design choice": "Released implementation", "Public evidence": f"{len(source_modules)} Python source modules", "Purpose": "Provides the concrete calculations"},
])
save_table(structure, "python_public_structure", "Public organization of RiskLabAI 3.2.0.", "tab:appendix-python-public-structure", fit=True)

public_area_table = pd.DataFrame({"Public area": public_areas})
save_table(public_area_table, "python_public_areas", "Root areas exposed by RiskLabAI 3.2.0.", "tab:appendix-python-public-areas")


,Design choice,Public evidence,Purpose
0,One public front door,RiskLabAI.__all__,Shows supported starting points
1,Modules loaded when requested,Module __getattr__,Delays optional and heavier imports
2,Grouped subject areas,12 exported areas,Keeps related methods together
3,Released implementation,144 Python source modules,Provides the concrete calculations


,Public area
0,backtest
1,causal_factor_analysis
2,cluster
3,controller
4,core
5,data
6,ensemble
7,features
8,hpc
9,optimization


,Public area
0,backtest
1,causal_factor_analysis
2,cluster
3,controller
4,core
5,data
6,ensemble
7,features
8,hpc
9,optimization


## Seeing the library as a public front door


In [3]:
readable = {
    "backtest": "Backtesting",
    "causal_factor_analysis": "Causal analysis",
    "cluster": "Clustering",
    "controller": "Older controllers",
    "core": "Core",
    "data": "Data",
    "ensemble": "Ensembles",
    "features": "Features",
    "hpc": "Parallel computing",
    "optimization": "Optimization",
    "pde": "PDE methods",
    "utils": "Utilities",
}
coordinates = [(-3.6, 2.7), (-1.8, 3.4), (0, 3.6), (1.8, 3.4), (3.6, 2.7), (4.2, 0.9), (4.2, -1.2), (2.8, -2.9), (0.95, -3.5), (-0.95, -3.5), (-2.8, -2.9), (-4.2, -0.7)]
topology = pd.DataFrame({"Public area": public_areas, "Label": [readable.get(name, name.replace("_", " ").title()) for name in public_areas], "X": [point[0] for point in coordinates], "Y": [point[1] for point in coordinates]})
topology.to_csv(FIGURE_DATA / "python_public_topology.csv", index=False)
figure, axis = plt.subplots(figsize=(9.2, 7.6), constrained_layout=True)
for index, row in topology.iterrows():
    axis.plot([0, row["X"]], [0, row["Y"]], color=PALETTE["light_gray"], linewidth=1.2)
    color = [PALETTE["blue"], PALETTE["teal"], PALETTE["orange"], PALETTE["magenta"], PALETTE["gray"]][index % 5]
    add_box(axis, row["X"], row["Y"], 1.85, 0.62, row["Label"], color, color, PALETTE["white"])
add_box(axis, 0, 0, 1.65, 0.82, "RiskLabAI\n3.2.0", PALETTE["near_white"], PALETTE["navy"])
axis.set(xlim=(-5.4, 5.4), ylim=(-4.35, 4.35), title="Public Entry Points in RiskLabAI 3.2.0")
axis.set_aspect("equal")
axis.axis("off")
save_figure(figure, "python_public_topology")


<Figure size 920x760 with 1 Axes>

## Tests


In [4]:
assert "def __getattr__" in root_source
assert public_areas
assert len(public_areas) == len(set(public_areas))
assert all((PYTHON_PACKAGE / name).exists() or name in root_source for name in public_areas)
assert len(source_modules) >= 100
